# 🏥 Lumbar Spinal Stenosis (LSS) AI Diagnostic & Benchmark Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssreeakash-cpu/ai-prooo/blob/main/LSS_Colab_Pipeline.ipynb)

This interactive notebook demonstrates the complete end-to-end clinical workflow:
1. **Medical DICOM MRI Ingestion & Spinal Canal Cropping** (L1/L2 to L5/S1)
2. **Patient Splitting ("No Cheating")** using `GroupShuffleSplit` on `study_id`
3. **Benchmarking 5 Image Models** (`EfficientNet-B3`, `DenseNet-121`, `Vision Transformer`, `YOLOv8 Classifier`, `U-Net Classifier`)
4. **Benchmarking 5 Tabular Models** (`TabNet`, `FT-Transformer`, `MLP`, `DeepFM`, `CNN-MLP`)
5. **Top 3 Selection & Head-to-Head Comparison** for both Image and Tabular modalities
6. **Multimodal Fusion Classification**
7. **Grad-CAM Visual Proof Heatmaps & 3-Panel PACS Diagnostic Views**

## 🛠️ Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository from GitHub if running in Google Colab
import os
if not os.path.exists('ai-prooo'):
    !git clone https://github.com/ssreeakash-cpu/ai-prooo.git
    %cd ai-prooo
else:
    %cd ai-prooo

# Install project dependencies
!pip install -q pydicom timm pytorch-tabnet ultralytics tabulate

## ⚡ Step 2: Check GPU Acceleration

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device:  {device.upper()}")
if device == "cuda":
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")

## 📊 Step 3: Generate High-Fidelity 512x512 RSNA DICOM MRI Dataset
Generates realistic Sagittal T2, Sagittal T1, and Axial T2 DICOM scans with lumbar lordosis curvature, authentic vertebral bodies (L1 through S1), thecal sac, and pathologically graded stenosis narrowing.

In [ ]:
from data.synthetic_data_generator import generate_mock_rsna_dataset
# Generate dataset for 20 patients with zero patient overlap
df_train, df_series, df_coords = generate_mock_rsna_dataset(num_patients=20)
print("\nSample labels dataframe head:")
df_train.head(3)

## 🏆 Step 4: Run Multi-Model Benchmark (5 Image & 5 Tabular Models)
Evaluates all 10 architectures, ranks them by clinical composite score, extracts the Top 3, performs head-to-head comparison, and trains the multimodal fusion network.

In [ ]:
from training.benchmark_runner import run_lss_benchmark

# Execute the benchmark suite
summary = run_lss_benchmark(num_epochs=2)

## 📈 Step 5: Visualize Top 3 Model Comparisons

In [ ]:
from IPython.display import Image, display
from pathlib import Path

img_plot = Path("outputs/reports/top3_image_models_comparison.png")
tab_plot = Path("outputs/reports/top3_tabular_models_comparison.png")

if img_plot.exists():
    print("--- Top 3 Image Models Head-to-Head Comparison ---")
    display(Image(str(img_plot)))

if tab_plot.exists():
    print("\n--- Top 3 Tabular Models Head-to-Head Comparison ---")
    display(Image(str(tab_plot)))

## 🔬 Step 6: Clinical Inference on Unseen Patient Scan & 3-Panel PACS View

In [ ]:
from inference.predict import LSSClinicalPredictor
from config.config import CONFIG
import matplotlib.pyplot as plt
import cv2

# Find a sample test DICOM scan
sample_files = list(CONFIG.raw_dicom_dir.glob("**/*.dcm"))
sample_dcm = sample_files[len(sample_files) // 2]

predictor = LSSClinicalPredictor(model_name="densenet_121")
finding = predictor.predict_slice(sample_dcm, level="L4/L5", generate_cam=True)

print("=" * 60)
print("  DIAGNOSTIC ASSESSMENT RESULT")
print("=" * 60)
print(f"Target Disc Level:   {finding['level']}")
print(f"Predicted Severity:  {finding['predicted_severity']}")
print(f"AI Confidence:       {finding['confidence'] * 100:.2f}%")
print("Probability Distribution:")
for grade, prob in finding['probabilities'].items():
    print(f"  - {grade:12s}: {prob * 100:.1f}%")
print("=" * 60)

# Display 3-Panel PACS Diagnostic Composite View
if 'composite_bgr' in finding:
    composite_rgb = cv2.cvtColor(finding['composite_bgr'], cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(18, 7), dpi=150)
    plt.imshow(composite_rgb)
    plt.axis('off')
    plt.title(f"PACS 3-Panel Diagnostic View: {finding['level']} ({finding['predicted_severity']})", fontsize=14, fontweight='bold')
    plt.show()

## 🌐 Step 7: Launch Interactive Web App in Google Colab (Optional)
You can run the web dashboard directly from Colab using localtunnel or pyngrok.

In [ ]:
# Run in background and expose via localtunnel
!pip install -q localtunnel
import subprocess
import time

p = subprocess.Popen(["python", "web/app.py"])
time.sleep(3)
print("Web server running. Open the public tunnel URL below:")
!npx localtunnel --port 5000